# 12. Prepare A2D2 — resume-safe v2

A2D2 `20190401_121727` Front Center + Bus를 Stage 3 v5용 10 Hz 데이터로 변환한다.

이 버전은 Colab 런타임 종료를 전제로 설계했다.
- TAR scan은 Google Drive checkpoint에서 재개
- index 완성 후에는 91.4 GiB TAR 전체 scan 재실행 없음
- 600-frame/60초 segment마다 checkpoint
- 완료 segment는 재실행 시 SKIP
- camera tar를 Google Drive에 풀지 않음
- GPU 불필요


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import subprocess, sys

DRIVE_ROOT = Path('/content/drive/MyDrive/Blackbox-Detection')
REPO = Path('/content/Blackbox-Detection')
BRANCH = 'stage3-sangchun'

if not REPO.exists():
    subprocess.run([
        'git','clone','-b',BRANCH,
        'https://github.com/sangchun1/Blackbox-Detection.git',
        str(REPO),
    ], check=True)
else:
    subprocess.run(['git','-C',str(REPO),'fetch','origin'], check=True)
    subprocess.run(['git','-C',str(REPO),'checkout',BRANCH], check=True)
    subprocess.run(['git','-C',str(REPO),'pull','--ff-only'], check=True)

SRC = REPO / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print('python:', sys.version)
print('repo  :', REPO)
print('src   :', SRC)


## 1. 패치 파일 적용 확인

이 notebook과 함께 제공한 `src/blackbox_detection/stage3/a2d2.py`를 repo 같은 경로에 덮어쓰고 commit/push한 뒤 실행한다.


In [ ]:
A2D2_MODULE = REPO / 'src/blackbox_detection/stage3/a2d2.py'
assert A2D2_MODULE.is_file(), A2D2_MODULE
print('a2d2.py:', A2D2_MODULE)
print('bytes  :', A2D2_MODULE.stat().st_size)


In [ ]:
from blackbox_detection.stage3.a2d2 import (
    A2D2PrepareConfig,
    load_bus_json,
    audit_bus,
    prepare_session_resume,
)
print('A2D2 resume module: PASS')


## 2. 데이터 경로 확인


In [ ]:
A2D2_ROOT = DRIVE_ROOT / 'DATASET/A2D2'
ARCHIVE_DIR = A2D2_ROOT / 'archives'
RAW_ROOT = A2D2_ROOT / 'raw'
PROCESSED_ROOT = A2D2_ROOT / 'processed'
SESSION_ID = '20190401_121727'

CAMERA_TAR = ARCHIVE_DIR / 'camera_lidar-20190401121727_camera_frontcenter.tar'
BUS_JSON = RAW_ROOT / 'camera_lidar' / SESSION_ID / 'bus' / '20190401121727_bus_signals.json'

assert CAMERA_TAR.is_file(), CAMERA_TAR
assert BUS_JSON.is_file(), BUS_JSON
print('camera tar GiB:', CAMERA_TAR.stat().st_size / 1024**3)
print('bus json MiB  :', BUS_JSON.stat().st_size / 1024**2)


## 3. 첫 v2 실행 전에만 old v1 산출물 정리 (선택)

기존 run은 index checkpoint를 남기지 못했으므로 재사용할 수 있는 핵심 결과가 없다.
단, 원본 `archives/`와 `raw/.../bus/`는 절대 삭제하지 않는다.

**v2 checkpoint가 한 번이라도 생긴 뒤에는 이 cleanup 셀을 다시 실행하지 않는다.**


In [ ]:
# 처음 v2로 갈아탈 때만 필요하면 실행.
# import shutil
# route = 'a2d2_20190401_121727'
# for p in [
#     PROCESSED_ROOT / 'videos' / route,
#     PROCESSED_ROOT / 'metadata' / route,
#     PROCESSED_ROOT / 'aux_metadata' / route,
#     PROCESSED_ROOT / 'done' / route,
#     PROCESSED_ROOT / 'cache' / route,
# ]:
#     if p.exists():
#         print('remove:', p)
#         shutil.rmtree(p)


## 4. Bus audit


In [ ]:
bus = load_bus_json(BUS_JSON)
r = audit_bus(bus)
print(r)
assert r['accel_dvdt_corr'] > 0.80
assert r['steer_yaw_corr'] > 0.50
print('BUS AUDIT: PASS')


## 5. Resume-safe full preparation

런타임이 끊기면 새 런타임에서 1, 2, 4, 5만 다시 실행한다.
`overwrite=False`를 유지해야 resume가 작동한다.


In [ ]:
import json
from pathlib import Path
cfg = A2D2PrepareConfig(
    processed_root=PROCESSED_ROOT,
    width=512,
    height=384,
    target_fps=10,
    segment_frames=600,
    crf=23,
    ffmpeg_preset='veryfast',
    max_camera_skew_ms=25.0,
    scan_checkpoint_every=2000,
    overwrite=False,
    work_root=Path('/content/a2d2_work'),
)

report = prepare_session_resume(
    camera_tar=CAMERA_TAR,
    bus_json=BUS_JSON,
    session_id=SESSION_ID,
    cfg=cfg,
)
print(json.dumps(report, indent=2))


## 6. Final audit


In [ ]:
import json, pandas as pd
from blackbox_detection.stage3.schema import read_frame_table

manifest = pd.read_csv(PROCESSED_ROOT / 'manifest.csv', dtype={'segment_id': str})
display(manifest)

print('segments:', len(manifest))
print('frames  :', int(manifest['num_frames'].sum()))
print('hours   :', float(manifest['num_frames'].sum() / 10 / 3600))

route = 'a2d2_20190401_121727'
for row in manifest.itertuples(index=False):
    sid = str(row.segment_id).zfill(3)
    video = PROCESSED_ROOT / row.video_relpath
    meta = PROCESSED_ROOT / row.metadata_relpath
    aux = PROCESSED_ROOT / row.aux_metadata_relpath
    done = PROCESSED_ROOT / 'done' / route / f'{sid}.json'
    assert video.is_file(), video
    assert meta.is_file(), meta
    assert aux.is_file(), aux
    assert done.is_file(), done
    df = read_frame_table(meta)
    assert len(df) == int(row.num_frames)
    assert df['frame_index_10hz'].tolist() == list(range(len(df)))

print('FINAL A2D2 PREP AUDIT: PASS')


## Resume 파일

```text
DATASET/A2D2/processed/
├── cache/a2d2_20190401_121727/
│   ├── camera_index.checkpoint.json   # scan 도중
│   └── camera_index.json              # scan 완료 후
├── done/a2d2_20190401_121727/
│   ├── 000.json
│   ├── 001.json
│   └── ...
├── videos/...
├── metadata/...
├── aux_metadata/...
├── manifest.csv
└── prepare_report.json
```
